# Drift–Skill and Error-Growth Attribution

This notebook investigates the predictability relationship between early forecast evolution (leads 1–3) and downstream prediction skill (leads 4–12, 13–24, and 4–24) in S2D hindcasts. It evaluates Niño3.4 and North Atlantic indices across five key variables (`TREFHT`, `SST`, `PSL`, `PRECT`, `H2OSOI`), comparing **JRA55_FOSIRL** and **Reanalysis** initialization.

Two complementary methodologies are presented:
1. **Method 1 (Baseline Early Drift)**: Relates mean change in absolute observation distance ($D_{\mathrm{early}} = \operatorname{mean}_{L=1..3}\Delta|e_{\mathrm{obs}}|$) to subsequent RMS error across initialization starts. Pearson correlation and quadrant counts evaluate whether starts that drift away early tend to fail later.
2. **Method 2 (Conditional Error-Growth Rate)**: Uses the OLS growth slope of absolute observation error ($G_{\mathrm{early}} = \operatorname{OLS\ slope}_{L=1..3}|e_{\mathrm{obs}}|$) and controls for initial condition error magnitude at lead 1 via conditional multiple regression $\log(E_{\mathrm{late}}) \sim G_{\mathrm{early}} + \log(|e_{L1}|)$. Circular moving-block bootstrap intervals, Spearman $\rho$, and leave-one-out predictive $R^2$ ensure robustness against confounding.

Both within-strategy relationships and paired strategy contrasts (FOSIRL minus Reanalysis for the same start year) are evaluated for each method.
Input preparation is automatic via `workflows.diagnostics.drift_inputs` with `DRIFT_INPUT_MODE="auto"`.


In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = ([Path(_repo_override).expanduser().resolve()] if _repo_override
                    else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
REPO_ROOT = next((p for p in _repo_candidates
                 if (p / "workflows" / "diagnostics" / "drift_inputs.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.drift_inputs import (
    DEFAULT_OUTPUT_ROOT, DEFAULT_FIGURE_ROOT, DEFAULT_VARIABLES,
    DEFAULT_INIT_MONTHS, DEFAULT_SOURCES, DEFAULT_SOURCE_LABELS,
    DEFAULT_SOURCE_COLORS, DEFAULT_SOURCE_MARKERS, DEFAULT_PLOT_REGIONS,
    DEFAULT_REGION_LABELS, DEFAULT_REGION_FILE_LABELS, MONTH_NAMES,
    load_regional_manifest, regional_product_path
)

%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import xarray as xr

from esp_lab.diagnostics.two_reference_drift import (
    bootstrap_block_mean_ci,
    bootstrap_paired_correlation_ci,
    characterize_conditional_relationship,
    compare_drift_skill_relationship,
    compare_early_error_growth_relationship,
    compute_early_drift_late_error,
    compute_early_error_growth,
    compute_early_error_growth_late_error,
)


## 1. Configuration


In [ ]:
OUTPUT_ROOT = DEFAULT_OUTPUT_ROOT
FIGURE_ROOT = DEFAULT_FIGURE_ROOT
VARIABLES = DEFAULT_VARIABLES
INIT_MONTHS = DEFAULT_INIT_MONTHS
SOURCES = DEFAULT_SOURCES
SOURCE_LABELS = DEFAULT_SOURCE_LABELS
SOURCE_COLORS = DEFAULT_SOURCE_COLORS
SOURCE_MARKERS = DEFAULT_SOURCE_MARKERS
PLOT_REGIONS = DEFAULT_PLOT_REGIONS
REGION_LABELS = DEFAULT_REGION_LABELS
REGION_FILE_LABELS = DEFAULT_REGION_FILE_LABELS

EARLY_LEADS = (1, 2, 3)
LATE_ERROR_WINDOWS = {
    'Months 4-12': range(4, 13),
    'Months 13-24': range(13, 25),
    'Months 4-24': range(4, 25),
}
BOOTSTRAP_BLOCK_LENGTH = 3
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 2024
BOOTSTRAP_CONFIDENCE = 0.95

PANEL_WIDTH = 3.3
PANEL_ASPECT_RATIO = 1.05
FIGURE_SCALE = 1.0
FIGURE_DPI = 300
FIGURE_PREFIX = 'fig_two_ref'
SHOW_FIGURES_INLINE = True
DRIFT_INPUT_MODE = os.environ.get('DRIFT_INPUT_MODE', 'auto')

FONT_SIZE = 9.0
TITLE_FONT_SIZE = 10.0
TICK_FONT_SIZE = 8.0
LEGEND_FONT_SIZE = 8.5
ANNOTATION_FONT_SIZE = 7.5
SCATTER_SIZE = 26
SCATTER_ALPHA = 0.85
PAIRED_SCATTER_SIZE = 30

def figure_size(width=PANEL_WIDTH, aspect_ratio=PANEL_ASPECT_RATIO, scale=FIGURE_SCALE):
    return (
        width * len(INIT_MONTHS) * scale,
        width * len(LATE_ERROR_WINDOWS) * aspect_ratio * scale,
    )

def lead_window_label(leads):
    leads = tuple(leads)
    for label, window_leads in LATE_ERROR_WINDOWS.items():
        if tuple(window_leads) == leads:
            return label
    return f'Leads {leads[0]}-{leads[-1]}'

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Load required regional fields

Load the required regional fields (`delta_abs_e_obs` and `e_obs`) from the precomputed regional products.


In [ ]:
regional_manifest = load_regional_manifest(
    OUTPUT_ROOT, VARIABLES, INIT_MONTHS, SOURCES, PLOT_REGIONS,
    mode=DRIFT_INPUT_MODE,
)

def load_drift_skill_inputs(variable, init_month, source):
    path = regional_product_path(variable, init_month, source, output_root=OUTPUT_ROOT)
    with xr.open_dataset(path) as opened:
        return opened[['e_obs', 'delta_abs_e_obs']].load()

drift_skill_inputs = {
    (variable, init_month, source): load_drift_skill_inputs(variable, init_month, source)
    for variable in VARIABLES
    for init_month in INIT_MONTHS
    for source in SOURCES
}
display(regional_manifest)
print(f'Loaded {len(drift_skill_inputs)} regional drift–skill inputs.')


## 3. Method 1: Mean Early Drift vs Later Skill (Baseline)

For initialization year $k$, early drift is

$$D_{\mathrm{early}}(k)=\operatorname{mean}_{L=1,2,3}\Delta|e_{\mathrm{obs}}|(k,L),$$

and later error is the RMS observation departure over the lead window shown in each panel. The annotation reports the Pearson correlation across initialization years. A positive correlation means that starts moving farther from observations early tend to have larger later errors.


In [ ]:
def finite_xy(relationship):
    x = np.asarray(relationship.early_drift).reshape(-1)
    y = np.asarray(relationship.late_error).reshape(-1)
    finite = np.isfinite(x) & np.isfinite(y)
    return x[finite], y[finite]


method1_figure_paths = []
correlation_rows = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(LATE_ERROR_WINDOWS), len(INIT_MONTHS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, late_leads in enumerate(LATE_ERROR_WINDOWS):
            for col, init_month in enumerate(INIT_MONTHS):
                ax = axes[row, col]
                for source in SOURCES:
                    fields = drift_skill_inputs[(
                        variable, init_month, source
                    )].sel(region=region_name)
                    relationship = compute_early_drift_late_error(
                        fields.delta_abs_e_obs, fields.e_obs,
                        early_leads=EARLY_LEADS, late_leads=late_leads,
                    )
                    x, y = finite_xy(relationship)
                    correlation = float(relationship.correlation)
                    ax.scatter(
                        x, y, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
                        marker=SOURCE_MARKERS[source],
                        color=SOURCE_COLORS[source],
                        edgecolor='white', linewidth=0.5,
                        label=(
                            f'{SOURCE_LABELS[source]} '
                            f'(r={correlation:+.2f})'
                        ),
                    )
                    correlation_rows.append({
                        'variable': variable, 'region': region_name,
                        'init_month': init_month, 'source': source,
                        'early_leads': lead_window_label(EARLY_LEADS),
                        'late_leads': lead_window_label(late_leads),
                        'correlation': correlation, 'n_starts': x.size,
                    })
                ax.axvline(0, color='0.35', linewidth=0.8, linestyle='--')
                ax.set_title(
                    f'{MONTH_NAMES[init_month]} initialization · '
                    f'error {lead_window_label(late_leads)}'
                )
                ax.set_ylabel('RMS observation error')
                ax.grid(True, alpha=0.25)
                ax.set_box_aspect(
                    PANEL_ASPECT_RATIO[1] / PANEL_ASPECT_RATIO[0]
                )
                ax.legend(frameon=False, loc='best')
        fig.suptitle(
            f'{variable}: observation-relative drift versus RMS error, '
            f'{REGION_LABELS[region_name]}'
        )
        fig.supxlabel(
            f'Mean Δ|e_obs|, {lead_window_label(EARLY_LEADS)} '
            '(negative = closer; positive = farther)'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_drift_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        method1_figure_paths.append(path)

correlation_table = pd.DataFrame(correlation_rows)
correlation_path = FIGURE_ROOT / (
    f'{FIGURE_PREFIX}_drift_skill_correlations.csv'
)
correlation_table.to_csv(correlation_path, index=False)
display(correlation_table.round({'correlation': 3}))
print(correlation_path)

## 4. Method 1: Paired Strategy Contrast

For the same initialization year $k$, this diagnostic compares

$$\Delta D_k=D_{\mathrm{FOSIRL},k}-D_{\mathrm{Reanalysis},k}$$

against

$$\Delta E_k=E_{\mathrm{FOSIRL},k}-E_{\mathrm{Reanalysis},k}.$$

The lower-left quadrant means FOSIRL has both smaller early drift and smaller later error; the upper-right means Reanalysis wins on both.


In [ ]:
method1_paired_paths = []
paired_rows = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(LATE_ERROR_WINDOWS), len(INIT_MONTHS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, late_leads in enumerate(LATE_ERROR_WINDOWS):
            for col, init_month in enumerate(INIT_MONTHS):
                ax = axes[row, col]
                relationships = {}
                for source in SOURCES:
                    fields = drift_skill_inputs[(
                        variable, init_month, source
                    )].sel(region=region_name)
                    relationships[source] = compute_early_drift_late_error(
                        fields.delta_abs_e_obs, fields.e_obs,
                        early_leads=EARLY_LEADS, late_leads=late_leads,
                    )

                paired = compare_drift_skill_relationship(
                    relationships['JRA55_FOSIRL'],
                    relationships['Reanalysis'],
                )
                interval = bootstrap_paired_correlation_ci(
                    paired.delta_drift, paired.delta_error,
                    n_bootstrap=N_BOOTSTRAP,
                    confidence=BOOTSTRAP_CONFIDENCE,
                    seed=BOOTSTRAP_SEED,
                )
                x = np.asarray(paired.delta_drift).reshape(-1)
                y = np.asarray(paired.delta_error).reshape(-1)
                finite = np.isfinite(x) & np.isfinite(y)
                x, y = x[finite], y[finite]
                correlation = float(paired.correlation)
                ci_lower = float(interval.ci_lower)
                ci_upper = float(interval.ci_upper)
                concordance = float(paired.concordance_fraction)

                ax.scatter(
                    x, y, s=PAIRED_SCATTER_SIZE, alpha=SCATTER_ALPHA,
                    color='tab:purple', edgecolor='white', linewidth=0.5,
                )
                if x.size >= 2 and not np.allclose(x, x[0]):
                    slope, intercept = np.polyfit(x, y, 1)
                    x_line = np.linspace(float(x.min()), float(x.max()), 100)
                    ax.plot(
                        x_line, slope * x_line + intercept,
                        color='tab:purple', linewidth=1.5,
                    )
                ax.axvline(0, color='0.25', linewidth=0.9, linestyle='--')
                ax.axhline(0, color='0.25', linewidth=0.9, linestyle='--')
                ax.text(
                    0.02, 0.97,
                    f'r={correlation:+.2f} '
                    f'[{ci_lower:+.2f}, {ci_upper:+.2f}]\n'
                    f'concordant={concordance:.0%}, n={x.size}',
                    transform=ax.transAxes, ha='left', va='top',
                    color='tab:purple', fontsize=ANNOTATION_FONT_SIZE,
                )
                ax.text(
                    0.02, 0.03, 'FOSIRL wins both',
                    transform=ax.transAxes, ha='left', va='bottom',
                    color='0.35', fontsize=ANNOTATION_FONT_SIZE - 1,
                )
                ax.text(
                    0.98, 0.97, 'Reanalysis wins both',
                    transform=ax.transAxes, ha='right', va='top',
                    color='0.35', fontsize=ANNOTATION_FONT_SIZE - 1,
                )
                ax.set_title(
                    f'{MONTH_NAMES[init_month]} initialization · '
                    f'error {lead_window_label(late_leads)}'
                )
                ax.set_ylabel('Δ RMS error (FOSIRL − Reanalysis)')
                ax.grid(True, alpha=0.25)
                ax.set_box_aspect(
                    PANEL_ASPECT_RATIO[1] / PANEL_ASPECT_RATIO[0]
                )

                paired_rows.append({
                    'variable': variable, 'region': region_name,
                    'init_month': init_month,
                    'early_leads': lead_window_label(EARLY_LEADS),
                    'late_leads': lead_window_label(late_leads),
                    'correlation': correlation,
                    'correlation_ci_lower': ci_lower,
                    'correlation_ci_upper': ci_upper,
                    'concordance_fraction': concordance,
                    'n_starts': x.size,
                })

        fig.suptitle(
            f'{variable}: paired drift–error contrast, '
            f'{REGION_LABELS[region_name]}'
        )
        fig.supxlabel(
            'Δ mean Δ|e_obs| (FOSIRL − Reanalysis)'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_paired_drift_error.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        method1_paired_paths.append(path)

paired_table = pd.DataFrame(paired_rows)
paired_table_path = FIGURE_ROOT / (
    f'{FIGURE_PREFIX}_paired_drift_error_correlations.csv'
)
paired_table.to_csv(paired_table_path, index=False)
display(paired_table.round({
    'correlation': 3, 'correlation_ci_lower': 3,
    'correlation_ci_upper': 3, 'concordance_fraction': 3,
}))
print(paired_table_path)

## 5. Method 2: Conditional Early Error-Growth Rate vs Later Skill

To control for initial condition error magnitude at lead 1, early error growth is modeled as the OLS slope over $L=1,2,3$:

$$G_{\mathrm{early}}(k)=\operatorname{OLS\ slope}_{L=1,2,3}|e_{\mathrm{obs}}(k,L)|,$$

and later error is the RMS observation departure over the lead window shown in each panel. The primary statistic is the standardized coefficient $\beta$ from $\log(E_{\mathrm{late}}) \sim G_{\mathrm{early}} + \log(|e_{L1}|)$, with a moving-block bootstrap confidence interval. Spearman $\rho$ and leave-one-year-out predictive $R^2$ are reported as sensitivity and predictive checks.


In [ ]:
def finite_xy(relationship):
    x = np.asarray(relationship.early_error_growth).reshape(-1)
    y = np.asarray(relationship.late_error).reshape(-1)
    baseline = np.asarray(relationship.baseline_error).reshape(-1)
    finite = (
        np.isfinite(x) & np.isfinite(y) & np.isfinite(baseline)
        & (y > 0) & (baseline > 0)
    )
    return x[finite], y[finite]


method2_figure_paths = []
correlation_rows = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(LATE_ERROR_WINDOWS), len(INIT_MONTHS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, late_leads in enumerate(LATE_ERROR_WINDOWS):
            for col, init_month in enumerate(INIT_MONTHS):
                ax = axes[row, col]
                for source in SOURCES:
                    fields = drift_skill_inputs[(
                        variable, init_month, source
                    )].sel(region=region_name)
                    relationship = compute_early_error_growth_late_error(
                        fields.e_obs,
                        early_leads=EARLY_LEADS, late_leads=late_leads,
                    )
                    x, y = finite_xy(relationship)
                    statistics = characterize_conditional_relationship(
                        relationship.early_error_growth,
                        relationship.baseline_error, relationship.late_error,
                        n_bootstrap=N_BOOTSTRAP,
                        confidence=BOOTSTRAP_CONFIDENCE,
                        block_length=BOOTSTRAP_BLOCK_LENGTH,
                        seed=BOOTSTRAP_SEED,
                    )
                    beta = float(statistics.standardized_slope)
                    beta_lower = float(statistics.standardized_slope_ci_lower)
                    beta_upper = float(statistics.standardized_slope_ci_upper)
                    rho = float(statistics.spearman_correlation)
                    cv_r2 = float(statistics.cross_validated_r2)
                    ax.scatter(
                        x, y, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
                        marker=SOURCE_MARKERS[source],
                        color=SOURCE_COLORS[source],
                        edgecolor='white', linewidth=0.5,
                        label=(
                            f'{SOURCE_LABELS[source]} '
                            f'(β={beta:+.2f}, ρ={rho:+.2f})'
                        ),
                    )
                    correlation_rows.append({
                        'variable': variable, 'region': region_name,
                        'init_month': init_month, 'source': source,
                        'early_leads': lead_window_label(EARLY_LEADS),
                        'late_leads': lead_window_label(late_leads),
                        'late_lead_midpoint': float(np.mean(late_leads)),
                        'standardized_slope': beta,
                        'standardized_slope_ci_lower': beta_lower,
                        'standardized_slope_ci_upper': beta_upper,
                        'spearman_correlation': rho,
                        'spearman_ci_lower': float(statistics.spearman_ci_lower),
                        'spearman_ci_upper': float(statistics.spearman_ci_upper),
                        'cross_validated_r2': cv_r2,
                        'n_starts': int(statistics.n_starts),
                    })
                ax.axvline(0, color='0.35', linewidth=0.8, linestyle='--')
                ax.set_title(
                    f'{MONTH_NAMES[init_month]} initialization · '
                    f'error {lead_window_label(late_leads)}'
                )
                ax.set_ylabel('RMS observation error')
                ax.grid(True, alpha=0.25)
                ax.set_box_aspect(
                    PANEL_ASPECT_RATIO[1] / PANEL_ASPECT_RATIO[0]
                )
                ax.legend(frameon=False, loc='best')
        fig.suptitle(
            f'{variable}: early error growth versus later RMS error, '
            f'{REGION_LABELS[region_name]}'
        )
        fig.supxlabel(
            f'Slope of |e_obs| versus lead, {lead_window_label(EARLY_LEADS)}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_growth_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        method2_figure_paths.append(path)

correlation_table = pd.DataFrame(correlation_rows)
correlation_path = FIGURE_ROOT / (
    f'{FIGURE_PREFIX}_conditional_growth_skill_statistics.csv'
)
correlation_table.to_csv(correlation_path, index=False)
display(correlation_table.round({
    'standardized_slope': 3, 'standardized_slope_ci_lower': 3,
    'standardized_slope_ci_upper': 3, 'spearman_correlation': 3,
    'spearman_ci_lower': 3, 'spearman_ci_upper': 3,
    'cross_validated_r2': 3,
}))
print(correlation_path)

effect_profile_paths = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            1, len(INIT_MONTHS), figsize=(11, 4.5),
            sharey=True, constrained_layout=True, squeeze=False,
        )
        for col, init_month in enumerate(INIT_MONTHS):
            ax = axes[0, col]
            for source in SOURCES:
                selected = correlation_table.loc[
                    correlation_table['variable'].eq(variable)
                    & correlation_table['region'].eq(region_name)
                    & correlation_table['init_month'].eq(init_month)
                    & correlation_table['source'].eq(source)
                ].sort_values('late_lead_midpoint')
                estimate = selected['standardized_slope'].to_numpy()
                errors = np.vstack((
                    estimate - selected['standardized_slope_ci_lower'].to_numpy(),
                    selected['standardized_slope_ci_upper'].to_numpy() - estimate,
                ))
                ax.errorbar(
                    selected['late_lead_midpoint'], estimate, yerr=errors,
                    marker=SOURCE_MARKERS[source], color=SOURCE_COLORS[source],
                    capsize=3, label=SOURCE_LABELS[source],
                )
            ax.axhline(0, color='0.35', linewidth=0.8, linestyle='--')
            ax.set_title(f'{MONTH_NAMES[init_month]} initialization')
            ax.set_xlabel('Later-window midpoint lead')
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
        axes[0, 0].set_ylabel('Conditional standardized slope β')
        fig.suptitle(
            f'{variable}: persistence of early-growth association, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_conditional_effect_profile.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        effect_profile_paths.append(path)

## 6. Method 2: Conditional Paired Strategy Contrast

For the same initialization year $k$, this diagnostic compares

$$\Delta G_k=G_{\mathrm{FOSIRL},k}-G_{\mathrm{Reanalysis},k}$$

against

$$\Delta\log E_k=\log E_{\mathrm{FOSIRL},k}-\log E_{\mathrm{Reanalysis},k}.$$

The conditional standardized slope controls for the paired difference in log L1 error. Spearman correlation and leave-one-year-out $R^2$ provide robustness and prediction checks.


In [ ]:
method2_paired_paths = []
paired_rows = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(LATE_ERROR_WINDOWS), len(INIT_MONTHS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, late_leads in enumerate(LATE_ERROR_WINDOWS):
            for col, init_month in enumerate(INIT_MONTHS):
                ax = axes[row, col]
                relationships = {}
                for source in SOURCES:
                    fields = drift_skill_inputs[(
                        variable, init_month, source
                    )].sel(region=region_name)
                    relationships[source] = compute_early_error_growth_late_error(
                        fields.e_obs,
                        early_leads=EARLY_LEADS, late_leads=late_leads,
                    )

                fosirl = relationships['JRA55_FOSIRL']
                reanalysis = relationships['Reanalysis']
                delta_growth = (
                    fosirl.early_error_growth - reanalysis.early_error_growth
                )
                delta_log_baseline = (
                    np.log(fosirl.baseline_error)
                    - np.log(reanalysis.baseline_error)
                )
                delta_log_error = (
                    np.log(fosirl.late_error) - np.log(reanalysis.late_error)
                )
                statistics = characterize_conditional_relationship(
                    delta_growth, delta_log_baseline, delta_log_error,
                    log_baseline=False, log_outcome=False,
                    n_bootstrap=N_BOOTSTRAP,
                    confidence=BOOTSTRAP_CONFIDENCE,
                    block_length=BOOTSTRAP_BLOCK_LENGTH,
                    seed=BOOTSTRAP_SEED,
                )
                mean_error_interval = bootstrap_block_mean_ci(
                    fosirl.late_error - reanalysis.late_error,
                    n_bootstrap=N_BOOTSTRAP,
                    confidence=BOOTSTRAP_CONFIDENCE,
                    block_length=BOOTSTRAP_BLOCK_LENGTH,
                    seed=BOOTSTRAP_SEED,
                )
                x = np.asarray(delta_growth).reshape(-1)
                y = np.asarray(delta_log_error).reshape(-1)
                z = np.asarray(delta_log_baseline).reshape(-1)
                finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
                x, y = x[finite], y[finite]
                beta = float(statistics.standardized_slope)
                beta_lower = float(statistics.standardized_slope_ci_lower)
                beta_upper = float(statistics.standardized_slope_ci_upper)
                rho = float(statistics.spearman_correlation)
                cv_r2 = float(statistics.cross_validated_r2)

                ax.scatter(
                    x, y, s=PAIRED_SCATTER_SIZE, alpha=SCATTER_ALPHA,
                    color='tab:purple', edgecolor='white', linewidth=0.5,
                )
                ax.axvline(0, color='0.25', linewidth=0.9, linestyle='--')
                ax.axhline(0, color='0.25', linewidth=0.9, linestyle='--')
                ax.text(
                    0.02, 0.97,
                    f'β={beta:+.2f} [{beta_lower:+.2f}, {beta_upper:+.2f}]\n'
                    f'ρ={rho:+.2f}, CV R²={cv_r2:+.2f}, '
                    f'n={int(statistics.n_starts)}\n'
                    f'mean ΔE={float(mean_error_interval.estimate):+.2g} '
                    f'[{float(mean_error_interval.ci_lower):+.2g}, '
                    f'{float(mean_error_interval.ci_upper):+.2g}]',
                    transform=ax.transAxes, ha='left', va='top',
                    color='tab:purple', fontsize=ANNOTATION_FONT_SIZE,
                )
                ax.text(
                    0.02, 0.03, 'FOSIRL wins both',
                    transform=ax.transAxes, ha='left', va='bottom',
                    color='0.35', fontsize=ANNOTATION_FONT_SIZE - 1,
                )
                ax.text(
                    0.98, 0.97, 'Reanalysis wins both',
                    transform=ax.transAxes, ha='right', va='top',
                    color='0.35', fontsize=ANNOTATION_FONT_SIZE - 1,
                )
                ax.set_title(
                    f'{MONTH_NAMES[init_month]} initialization · '
                    f'error {lead_window_label(late_leads)}'
                )
                ax.set_ylabel('Δ log RMS error (FOSIRL − Reanalysis)')
                ax.grid(True, alpha=0.25)
                ax.set_box_aspect(
                    PANEL_ASPECT_RATIO[1] / PANEL_ASPECT_RATIO[0]
                )

                paired_rows.append({
                    'variable': variable, 'region': region_name,
                    'init_month': init_month,
                    'early_leads': lead_window_label(EARLY_LEADS),
                    'late_leads': lead_window_label(late_leads),
                    'late_lead_midpoint': float(np.mean(late_leads)),
                    'standardized_slope': beta,
                    'standardized_slope_ci_lower': beta_lower,
                    'standardized_slope_ci_upper': beta_upper,
                    'spearman_correlation': rho,
                    'spearman_ci_lower': float(statistics.spearman_ci_lower),
                    'spearman_ci_upper': float(statistics.spearman_ci_upper),
                    'cross_validated_r2': cv_r2,
                    'mean_delta_late_error': float(mean_error_interval.estimate),
                    'mean_delta_late_error_ci_lower': float(mean_error_interval.ci_lower),
                    'mean_delta_late_error_ci_upper': float(mean_error_interval.ci_upper),
                    'n_starts': int(statistics.n_starts),
                })

        fig.suptitle(
            f'{variable}: paired early-growth and later-error contrast, '
            f'{REGION_LABELS[region_name]}'
        )
        fig.supxlabel(
            'Δ slope of |e_obs| versus lead (FOSIRL − Reanalysis)'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_paired_growth_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        method2_paired_paths.append(path)

paired_table = pd.DataFrame(paired_rows)
paired_table_path = FIGURE_ROOT / (
    f'{FIGURE_PREFIX}_paired_conditional_growth_skill_statistics.csv'
)
paired_table.to_csv(paired_table_path, index=False)
display(paired_table.round({
    'standardized_slope': 3, 'standardized_slope_ci_lower': 3,
    'standardized_slope_ci_upper': 3, 'spearman_correlation': 3,
    'spearman_ci_lower': 3, 'spearman_ci_upper': 3,
    'cross_validated_r2': 3, 'mean_delta_late_error': 3,
    'mean_delta_late_error_ci_lower': 3,
    'mean_delta_late_error_ci_upper': 3,
}))
print(paired_table_path)

paired_effect_profile_paths = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            1, len(INIT_MONTHS), figsize=(11, 4.5),
            sharey=True, constrained_layout=True, squeeze=False,
        )
        for col, init_month in enumerate(INIT_MONTHS):
            ax = axes[0, col]
            selected = paired_table.loc[
                paired_table['variable'].eq(variable)
                & paired_table['region'].eq(region_name)
                & paired_table['init_month'].eq(init_month)
            ].sort_values('late_lead_midpoint')
            estimate = selected['standardized_slope'].to_numpy()
            errors = np.vstack((
                estimate - selected['standardized_slope_ci_lower'].to_numpy(),
                selected['standardized_slope_ci_upper'].to_numpy() - estimate,
            ))
            ax.errorbar(
                selected['late_lead_midpoint'], estimate, yerr=errors,
                marker='o', color='tab:purple', capsize=3,
            )
            ax.axhline(0, color='0.35', linewidth=0.8, linestyle='--')
            ax.set_title(f'{MONTH_NAMES[init_month]} initialization')
            ax.set_xlabel('Later-window midpoint lead')
            ax.grid(True, alpha=0.25)
        axes[0, 0].set_ylabel('Paired conditional standardized slope β')
        fig.suptitle(
            f'{variable}: persistence of paired growth–skill association, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_paired_effect_profile.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        paired_effect_profile_paths.append(path)

## 7. Output validation


In [ ]:
expected_figure_count = sum(len(PLOT_REGIONS[variable]) for variable in VARIABLES)
print(f'Method 1 figures: {len(method1_figure_paths)} within-strategy, {len(method1_paired_paths)} paired.')
print(f'Method 2 figures: {len(method2_figure_paths)} within-strategy, {len(method2_paired_paths)} paired.')
assert len(method1_figure_paths) == expected_figure_count
assert len(method1_paired_paths) == expected_figure_count
assert len(method2_figure_paths) == expected_figure_count
assert len(method2_paired_paths) == expected_figure_count
print('All Method 1 and Method 2 figures validated successfully!')
